# Forecasting de ventas 2025

Este notebook carga los datos de inferencia de ventas para 2025 y prepara el DataFrame para su análisis y forecasting.

## 1. Importar librerías necesarias

Importamos las mismas librerías utilizadas en el notebook de entrenamiento para el procesamiento y análisis de datos.

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
import holidays

## 2. Cargar archivo de inferencias en DataFrame

Cargamos el archivo `ventas_2025_inferencia.csv` ubicado en `data/raw/inferencia` en un DataFrame llamado `inferencia_df`.

In [43]:
inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')

# Mostrar las primeras filas para verificar la carga
display(inferencia_df.head())

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [44]:
inferencia_df.shape

(888, 13)

In [45]:
# ...existing code...
# filepath: notebooks/forecasting.ipynb

# 1. Conversión de fecha y creación de variables temporales
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.weekday
inferencia_df['nombre_mes'] = inferencia_df['fecha'].dt.month_name(locale='es_ES')
inferencia_df['nombre_dia_semana'] = inferencia_df['fecha'].dt.day_name(locale='es_ES')
inferencia_df['es_fin_de_semana'] = inferencia_df['dia_semana'].isin([5,6])
inferencia_df['dia_del_año'] = inferencia_df['fecha'].dt.dayofyear
inferencia_df['semana_del_año'] = inferencia_df['fecha'].dt.isocalendar().week

# 2. Variables de festivos y eventos especiales
es_festivo = []
es_Black_Friday = []
es_Cyber_Monday = []
es_laborable = []

for fecha in inferencia_df['fecha']:
    festivos = holidays.country_holidays('ES', years=[fecha.year])
    es_festivo.append(fecha in festivos)
    # Black Friday: último viernes de noviembre
    black_friday = max([d for d in pd.date_range(start=f'{fecha.year}-11-01', end=f'{fecha.year}-11-30') if d.weekday() == 4])
    es_Black_Friday.append(fecha == black_friday)
    # Cyber Monday: lunes siguiente al Black Friday
    cyber_monday = black_friday + pd.Timedelta(days=3)
    es_Cyber_Monday.append(fecha == cyber_monday)
    es_laborable.append((fecha not in festivos) and (fecha.weekday() < 5))

inferencia_df['es_festivo'] = es_festivo
inferencia_df['es_Black_Friday'] = es_Black_Friday
inferencia_df['es_Cyber_Monday'] = es_Cyber_Monday
inferencia_df['es_laborable'] = es_laborable

# 3. Variables de descuento
inferencia_df['descuento_pct'] = 1 - (inferencia_df['precio_venta'] / inferencia_df['precio_base'])

# 4. Precio competencia y ratio_precio
competidores = ['Amazon', 'Decathlon', 'Deporvillage']
if all(col in inferencia_df.columns for col in competidores):
    inferencia_df['precio_competencia'] = inferencia_df[competidores].mean(axis=1)
    inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']
    inferencia_df = inferencia_df.drop(columns=competidores)
else:
    print('⚠️ Alguna columna de competidor no existe en el DataFrame.')

# 5. Variables lag y media móvil (rellenadas con NaN, ya que no hay histórico previo)
for lag in range(1,8):
    inferencia_df[f'unidades_vendidas_lag{lag}'] = np.nan
inferencia_df['unidades_vendidas_mm7'] = np.nan

# 6. One Hot Encoding para nombre, categoria y subcategoria
for col in ['nombre', 'categoria', 'subcategoria']:
    if col in inferencia_df.columns:
        inferencia_df[col + '_H'] = inferencia_df[col]

cols_h = [c for c in ['nombre_H', 'categoria_H', 'subcategoria_H'] if c in inferencia_df.columns]
inferencia_df = pd.get_dummies(inferencia_df, columns=cols_h, drop_first=False)

# 7. Asegurar que las columnas coinciden con las del modelo (ajustar orden y añadir columnas faltantes con 0)
import pandas as pd
df_modelo = pd.read_csv('../data/processed/df.csv', nrows=1)
columnas_modelo = df_modelo.columns.tolist()
for col in columnas_modelo:
    if col not in inferencia_df.columns:
        inferencia_df[col] = 0
inferencia_df = inferencia_df[columnas_modelo]

# 8. Filtrar solo registros de noviembre
inferencia_df = inferencia_df[inferencia_df['mes'] == 11]

# 9. Guardar el DataFrame transformado
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)
print('🤖 DataFrame de inferencia transformado y guardado en ../data/processed/inferencia_df_transformado.csv')
# ...existing code...

🤖 DataFrame de inferencia transformado y guardado en ../data/processed/inferencia_df_transformado.csv


In [46]:
inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_H_Esterilla Yoga,subcategoria_H_Mancuernas Ajustables,subcategoria_H_Mochila Trekking,subcategoria_H_Pesa Rusa,subcategoria_H_Pesas Casa,subcategoria_H_Rodillera Yoga,subcategoria_H_Ropa Montaña,subcategoria_H_Ropa Running,subcategoria_H_Zapatillas Running,subcategoria_H_Zapatillas Trail
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
169,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,False,False,False,False,False,False,False,False,True,False
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,False,False,False,False,False,False,False,False,True,False
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,False,False,False,False,False,False,False,False,True,False
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,False,False,False,False,False,False,False,True,False,False


In [47]:
inferencia_df.describe()


,fecha,precio_base,unidades_vendidas,precio_venta,ingresos,año,dia_semana,mes,dia_mes,dia_del_año,...,unidades_vendidas_lag2,unidades_vendidas_lag3,unidades_vendidas_lag4,unidades_vendidas_lag5,unidades_vendidas_lag6,unidades_vendidas_lag7,unidades_vendidas_mm7,descuento_pct,precio_competencia,ratio_precio
count,720,720.000000,0.0,720.000000,0.0,720.0,720.000000,720.0,720.000000,720.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,720.000000,720.000000,720.000000
mean,2025-11-15 12:00:00,123.125000,NaN,122.500681,NaN,2025.0,3.166667,11.0,15.500000,319.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004946,116.383495,1.042318
min,2025-11-01 00:00:00,20.000000,NaN,17.000000,NaN,2025.0,0.000000,11.0,1.000000,305.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.019778,18.473333,0.871923
25%,2025-11-08 00:00:00,48.750000,NaN,45.877500,NaN,2025.0,1.000000,11.0,8.000000,312.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.006250,46.133333,1.015508
50%,2025-11-15 12:00:00,72.500000,NaN,71.315000,NaN,2025.0,3.000000,11.0,15.500000,319.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,69.930000,1.040183
75%,2025-11-23 00:00:00,118.750000,NaN,115.000000,NaN,2025.0,5.000000,11.0,23.000000,327.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007600,110.873333,1.065829
max,2025-11-30 00:00:00,830.000000,NaN,830.000000,NaN,2025.0,6.000000,11.0,30.000000,334.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.150000,837.193333,1.203810
std,NaN,165.668531,NaN,164.932155,NaN,0.0,2.035840,0.0,8.661458,8.661458,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.028752,155.641301,0.047263


In [48]:
inferencia_df.shape

(720, 78)

In [49]:
inferencia_df.producto_id.nunique()

24

In [50]:
inferencia_df.fecha.nunique()

30

In [51]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'año', 'dia_semana', 'mes', 'nombre_mes', 'dia_mes',
       'nombre_dia_semana', 'es_fin_de_semana', 'es_festivo',
       'es_Black_Friday', 'es_Cyber_Monday', 'es_laborable', 'dia_del_año',
       'semana_del_año', 'unidades_vendidas_lag1', 'unidades_vendidas_lag2',
       'unidades_vendidas_lag3', 'unidades_vendidas_lag4',
       'unidades_vendidas_lag5', 'unidades_vendidas_lag6',
       'unidades_vendidas_lag7', 'unidades_vendidas_mm7', 'descuento_pct',
       'precio_competencia', 'ratio_precio',
       'nombre_H_Adidas Own The Run Jacket', 'nombre_H_Adidas Ultraboost 23',
       'nombre_H_Asics Gel Nimbus 25', 'nombre_H_Bowflex SelectTech 552',
       'nombre_H_Columbia Silver Ridge',
       'nombre_H_Decathlon Bandas Elásticas Set', 'nombre_H_Domyos BM900',
       'nombre_H_Domyos Kit Mancuernas 20kg',
       'nombre_H_